# P06 · lidocaine × polarity, 28-09-2026

**Why this notebook exists.** P06 is the only participant whose lidocaine session was run in
**all three polarities**. The across-participants notebook
(`P04_P05_lidocaine_30Hz_vs_arcex.ipynb`) uses P06's **cathodic** blocks so the three people can
be compared on the montage they share. Here the question is one only P06 can answer:

> **Does the skin block cost the same whichever way the field runs?**

If cathodic and anodic recruit different fibres — which the polarity notebooks suggest — then
removing cutaneous afferents should not cost them equally. Biphasic sits between the two and is
the check on that reading.

**12 recordings: 2 blocks × 3 polarities × 2 protocols.** Electrode 2, anode at the iliac crests,
lidocaine applied at **15:55**.

> The session log writes polarity as **positive / negative**. **Negative is cathodic**, positive
> is anodic — confirmed by the experimenter and by two notes in the log itself.

> The data folder is named `28-08-2026`, but every filename and the log say **28-09-2026**.

**The rule for intensity.** Each panel — one protocol and one polarity — takes the threshold
picked in **its own `before lidocaine`** recording, and both of its blocks are read at that one
intensity, muscle by muscle. Nothing crosses between protocols or polarities. If the lidocaine
block were read at its own threshold, a change in the response and a change in the current would
be indistinguishable.

**Colours:** 30 Hz **grey**, ARC-EX **red**, biphasic **blue**; pale = before, solid = with
lidocaine.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       condition_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale,
                       fig_threshold_grid, fig_detection_grid,
                       fig_group_summary, snr_verdict, fig_noise_check)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D = "tSCS_CHUV_data/28-08-2026/"      # P06, 28-09-2026, electrode 2.  Lidocaine at 15:55.

ALL = {   # (protocol, polarity, block) -> csv.  "negative" in the log = cathodic.
 ("30 Hz",  "cathodic", "before"):    D + "Burst_autosave_20260928_152710_593ms.csv",       # 15:27
 ("30 Hz",  "cathodic", "lidocaine"): D + "Burst_autosave_20260928_165756_941ms.csv",       # 16:57
 ("30 Hz",  "anodic",   "before"):    D + "Burst_autosave_20260928_152456_242ms.csv",       # 15:24
 ("30 Hz",  "anodic",   "lidocaine"): D + "Burst_autosave_20260928_165525_067ms.csv",       # 16:55
 ("30 Hz",  "biphasic", "before"):    D + "Burst_Biphasic_autosave_20260928_152834_294ms.csv",
 ("30 Hz",  "biphasic", "lidocaine"): D + "Burst_Biphasic_autosave_20260928_165942_870ms.csv",
 ("ARC-EX", "cathodic", "before"):    D + "Modulated_autosave_20260928_154023_427ms.csv",   # 15:40
 ("ARC-EX", "cathodic", "lidocaine"): D + "Modulated_autosave_20260928_170517_985ms.csv",   # 17:05
 ("ARC-EX", "anodic",   "before"):    D + "Modulated_autosave_20260928_153621_340ms.csv",   # 15:36
 ("ARC-EX", "anodic",   "lidocaine"): D + "Modulated_autosave_20260928_170249_644ms.csv",   # 17:02
 ("ARC-EX", "biphasic", "before"):    D + "Modulated_Biphasic_autosave_20260928_154333_773ms.csv",
 ("ARC-EX", "biphasic", "lidocaine"): D + "Modulated_Biphasic_autosave_20260928_170744_182ms.csv",
}

# ======== what to analyse ========
PROTOCOLS  = ["30 Hz", "ARC-EX"]
POLARITIES = ["cathodic", "anodic", "biphasic"]      # drop any you do not want
# =================================

CONTROL    = "before"                      # the block everything is measured against
PHASES     = [CONTROL, "lidocaine"]
PANELS     = [f"{p} · {pol}" for p in PROTOCOLS for pol in POLARITIES]
RUN        = {(f"{p} · {pol}", ph): ALL[(p, pol, ph)]
              for p in PROTOCOLS for pol in POLARITIES for ph in PHASES}

# colour = protocol, shade = block; biphasic gets its own hue
PCOL  = {"30 Hz": "#8C8C8C", "ARC-EX": "#C0392B", "biphasic": "#2E6F95"}
BAR_COLOUR = {CONTROL: "#C6C6C6",
              "lidocaine · 1st pulse": "#E3A9A1", "lidocaine · mean 1-10": "#C0392B"}

# ======== the signal-to-noise gate ========
SNR_GATE = None    # None = draw every reading. 2.0 = leave out anything under twice its noise.
SNR_SHOW = 2.0     # still MARKS anything under this, and lists it for the noise check below
# ==========================================

MUSCLES = ["Flex. digitorum (L)", "Ext. digitorum (L)",      # left arm, then right
           "Flex. digitorum (R)", "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN
for pan in PANELS:
    for ph in PHASES:
        f = runs[(pan, ph)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:20s} {ph:10s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA, "
              f"step {amps[1] - amps[0]:g}" + ("   <- control" if ph == CONTROL else ""))

## 2 · The motor thresholds you chose

The threshold is picked in each panel's **`before`** recording and used for **both** of that
panel's blocks, muscle by muscle. A panel is one protocol and one polarity, so all six carry their
own thresholds.

Two grids: **what is saved for each recording**, read straight from
`results/P06_2026-09-28_polarity_lidocaine/mt_*.csv`, and below the numbers **what the analysis
uses** — one marked sweep per panel, that panel's `before` threshold, drawn on its `lidocaine` row
as well.

The `lidocaine` picks set no intensity. They are the first result in their own right — how far the
threshold moved — and section 4a reads them.

In [ ]:
CHECK = {f"{pan} · {ph}": runs[(pan, ph)] for pan in PANELS for ph in PHASES}

picked = fig_threshold_grid(CHECK, MUSCLES, title="P06 · 28-09 — what is saved for each recording")

### The numbers, and where each one came from

In [ ]:
MT = {pan: thresholds_for(runs[(pan, CONTROL)], MUSCLES, consecutive=2, **KW) for pan in PANELS}
for pan in PANELS:
    print(f"{pan:20s} {CONTROL:10s} {threshold_source(runs[(pan, CONTROL)])}")

MT_ALL = {lab: picked.get(lab, {}) for lab in CHECK}

USED = {f"{pan} · {ph}": MT[pan] for pan in PANELS for ph in PHASES}
_ = fig_threshold_grid(CHECK, MUSCLES, thresholds=USED,
                       title=f"the motor threshold used: each panel's {CONTROL}")

print("\nmotor threshold (mA), as picked in each recording")
print(f"{'recording':34s}" + "".join(f"{m.split(' (')[0][:8] + m[-3:]:>16s}" for m in MUSCLES))
for lab in CHECK:
    print(f"{lab:34s}" + "".join(f"{str(MT_ALL[lab].get(m) or '-'):>16s}" for m in MUSCLES))

MT_STAMP = mt_stamp(runs)

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2, showing how the peaks are found **at the intensity section 4
measures** — every block of a panel at that panel's `before` threshold.

| | |
|---|---|
| **green** | the response window |
| **red bands** | the only places a peak may be taken, `ANCHOR_WIN_MS` either side of the train's own average latency |
| red ring | flagged — on a window border, or at a different latency |
| corner | `clean`, or how many of the 10 pulses were flagged |

`TRY` loosens the settings for this figure only; copy anything you keep into `KW` in section 1.

In [ ]:
TRY = dict(KW)
# TRY["anchor_win_ms"] = 6.0
# TRY["anchor"] = None
# TRY["min_snr"] = None

DET_MT = {f"{pan} · {ph}": MT[pan] for pan in PANELS for ph in PHASES}
fig_detection_grid(CHECK, DET_MT, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title=f"peak detection at each panel's {CONTROL} threshold", **TRY)

## 4 · The results

### 4a · How far lidocaine moved the threshold

The threshold picked **with lidocaine** as a % of the one picked **before**, same muscle, same
panel. **100 % = the skin block cost that polarity nothing.** Above 100 % means it needed more
current once cutaneous afferents were blocked.

The comparison to make is **between polarities within a protocol**: if one polarity's threshold
rises and another's does not, they were not recruiting the same thing.

In [ ]:
shift = {}
for pan in PANELS:
    b, a = MT_ALL[f"{pan} · {CONTROL}"], MT_ALL[f"{pan} · lidocaine"]
    for m in MUSCLES:
        if b.get(m) and a.get(m):
            shift[(pan, "lidocaine", m)] = dict(pct=100 * a[m] / b[m], before=b[m], after=a[m])

fig_vibration(shift, PANELS, MUSCLES, ["lidocaine"], colours={"lidocaine": "#C0392B"},
              ylabel="threshold, % of before", ref=100, ylim=(0, 200),
              titles={pan: pan for pan in PANELS},
              title="P06 · 28-09 — how far lidocaine moved the motor threshold")

print(f"{'panel':20s}{'muscle':22s}{'before':>9s}{'lido':>9s}{'%':>7s}")
for pan in PANELS:
    for m in MUSCLES:
        v = shift.get((pan, "lidocaine", m))
        print(f"{pan:20s}{m:22s}"
              + (f"{v['before']:>7g} mA{v['after']:>7g} mA{v['pct']:>6.0f} %" if v
                 else f"{'-':>9s}{'-':>9s}{'-':>7s}"))

### 4b · The response at a matched intensity

**Three bars per muscle.** The pale bar is `before` at 100 %. Beside it, `lidocaine` measured two
ways, both as a % of `before` measured the same way: its **1st pulse**, and the **mean of all 10
pulses**. They are not the same number and a manipulation can move one without the other.

A bar at 0 means the response was there before and gone with lidocaine; the axis says
`under its noise` or `< 2x noise` beside it. **Neither label says there was no response** — it says
what was measured against that recording's own noise. The by-eye check below settles which.

In [ ]:
warn_if_stale(runs, MT_STAMP, used=[(pan, CONTROL) for pan in PANELS])

tab = condition_table(runs, MT, MUSCLES, PHASES, base=CONTROL, match="common", **KW)

BARS = [CONTROL, "lidocaine · 1st pulse", "lidocaine · mean 1-10"]
both = {}
for pan in PANELS:
    for m in MUSCLES:
        if (pan, CONTROL, m) in tab:
            both[(pan, CONTROL, m)] = dict(tab[(pan, CONTROL, m)], pct=100.0)
        v = tab.get((pan, "lidocaine", m))
        if v:
            both[(pan, "lidocaine · 1st pulse", m)] = dict(v, pct=v.get("pct"))
            both[(pan, "lidocaine · mean 1-10", m)] = dict(v, pct=v.get("pct_all"))

fig_vibration(both, PANELS, MUSCLES, BARS, colours=BAR_COLOUR,
              min_snr_ratio=SNR_GATE, zero_if_absent=True,
              ylabel=f"% of {CONTROL}", ylim=(0, 200),
              titles={pan: pan for pan in PANELS},
              title=f"P06 · 28-09 — lidocaine as % of {CONTROL}: 1st pulse and whole train",
              save=None)

### The numbers, so they can be checked

In [ ]:
print(f"{'panel':20s}{'muscle':22s}{'mA':>5s}"
      + "".join(f"{ph[:10] + ' 10p':>16s}{ph[:10] + ' 1st':>16s}" for ph in PHASES))
for pan in PANELS:
    for m in MUSCLES:
        amp = next((tab[(pan, ph, m)]["amp"] for ph in PHASES if (pan, ph, m) in tab), None)
        if amp is None:
            continue
        cells = []
        for ph in PHASES:
            v, why = snr_verdict(tab, (pan, ph, m), CONTROL, SNR_GATE)
            if why:
                cells += [why, ""]
            elif v is None:
                cells += ["-", "-"]
            else:
                mk = " ?" if v.get("snr", 9e9) < SNR_SHOW else "  "
                cells += [f"{v['pct_all']:.0f} %" + mk, f"{v['pct']:.0f} %" + mk]
        print(f"{pan:20s}{m:22s}{amp:5g}" + "".join(f"{c_:>16s}" for c_ in cells))
print(f"\n10p = mean of all 10 pulses   1st = the 1st pulse only, both as % of {CONTROL}")
print(f"?  = under {SNR_SHOW:g}x its own noise - drawn, but check it in the noise panels below")

### Is it noise, or is there nothing there? — decide by eye

For every reading marked `?`, the response windows beside windows of the same length from the
90 ms **before** the stimulus, on one scale. Alike means noise. A deflection at a consistent
latency that the pre-stimulus windows do not have means a response, however small the ratio.

In [ ]:
ONLY = None   # None = every marked reading; or ("ARC-EX · anodic", "lidocaine", "Ext. digitorum (L)")

flagged = [(pan, ph, m) for pan in PANELS for ph in PHASES for m in MUSCLES
           if (pan, ph, m) in tab and tab[(pan, ph, m)].get("snr", 9e9) < SNR_SHOW]
todo = [ONLY] if ONLY else flagged
print(f"{len(flagged)} reading(s) under {SNR_SHOW:g}x noise"
      + ("  (the gate is OFF - all of them are in the figures)" if SNR_GATE is None else "") + "\n")
for pan, ph, m in todo:
    fig_noise_check(runs[(pan, ph)], m, tab[(pan, ph, m)].get("amp"),
                    title=f"{pan} · {ph} · {m}", **KW)

### 4c · One figure per muscle, with the EMG

The 1st pulse in mV, the 2nd and the mean of 2–10 as a % of it, and the EMG underneath.
Millivolts **are** comparable here — one participant, one session, one electrode — so the mV panel
can be read across panels as well as within them.

In [ ]:
MT_cond = {(pan, ph): {m: tab[(pan, ph, m)]["amp"] for m in MUSCLES if (pan, ph, m) in tab}
           for pan in PANELS for ph in PHASES}

for m in MUSCLES:
    fig_bars(runs, MT_cond, m, PANELS, protocols=tuple(PHASES), steps=0,
             colours={CONTROL: "#8C8C8C", "lidocaine": "#C0392B"},
             names={ph: ph for ph in PHASES},
             min_snr_ratio=SNR_GATE, rest_from=2, with_p1=True, traces=True,
             title=m, save=None, **KW)

### 4d · Pooled over the panels

The same numbers averaged: for each muscle the **mean over the six panels** with **SD**, and every
panel as a dot. It answers whether lidocaine does the same thing regardless of protocol and
polarity — a tight bar means it does, a wide one means the panels disagree and the per-panel
figure above is where the result is.

In [ ]:
for key, lab in (("pct_all", f"% of {CONTROL} (mean 1-10)"),
                ("pct", f"% of {CONTROL} (1st pulse)")):
    fig_group_summary(tab, PANELS, MUSCLES, ["lidocaine"], base={"lidocaine": CONTROL}, key=key,
                      colours={"lidocaine": "#C0392B"}, min_snr_ratio=SNR_GATE, err="sd",
                      reference=(CONTROL, 100), ylabel=lab, ref=100, ylim=(0, 200),
                      title=f"Lidocaine as % of before — {lab}", save=None)

## 5 · Change a threshold

**Six cells** — each panel's `before` recording, the control everything from 4b onward is measured
against. Move the slider, press **Set to slider**, then **Save**, and re-run from section 2.

The `lidocaine` picks feed **4a** and section 2's grid; they set no intensity. To pick one, call
`repick(("30 Hz · cathodic", "lidocaine"))` in a new cell.

In [ ]:
def repick(key):
    """Open the picker for one recording, with a Save button.

    The green line is the DETECTOR's answer for this recording - not another block's, and not
    whatever is already saved (that is the orange trace)."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('30 Hz · cathodic', 'before'))

In [ ]:
repick(('30 Hz · anodic', 'before'))

In [ ]:
repick(('30 Hz · biphasic', 'before'))

In [ ]:
repick(('ARC-EX · cathodic', 'before'))

In [ ]:
repick(('ARC-EX · anodic', 'before'))

In [ ]:
repick(('ARC-EX · biphasic', 'before'))